In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

In [ ]:
from pathlib import Path
DATA_DIR = Path("../data/statsbomb")
list((DATA_DIR).glob("*"))

In [ ]:
comps = pd.read_json("../data/statsbomb/competitions.json")
comps.head()

In [ ]:
comps = pd.read_json(DATA_DIR / "competitions.json")

# Build list of all (comp_id, season_id) pairs that have a match file
available = []
for comp_dir in sorted((DATA_DIR / "matches").iterdir()):
    for season_file in sorted(comp_dir.glob("*.json")):
        available.append((int(comp_dir.name), int(season_file.stem)))

print(f"Found {len(available)} competition/season pairs")
available[:5]

In [ ]:
all_events = []
missing_events = []

events_dir = DATA_DIR / "events"

for comp_id, season_id in available:
    match_file = DATA_DIR / "matches" / str(comp_id) / f"{season_id}.json"
    matches = pd.read_json(match_file)

    for match_id in matches["match_id"]:
        event_file = events_dir / f"{match_id}.json"
        if not event_file.exists():
            missing_events.append(match_id)
            continue
        with open(event_file, "r", encoding="utf-8") as f:
            events = json.load(f)
            df = pd.json_normalize(events, sep="_")
            df["match_id"] = match_id
            all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
print(f"Loaded {len(all_events)} matches, {len(events_df)} events total")
print(f"Skipped {len(missing_events)} matches with no event file")

In [ ]:
shots = events_df[events_df["type_name"] == "Shot"].copy()
shots.head()

In [ ]:
shots["shot_x"] = shots["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
shots["shot_y"] = shots["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

shots["goal"] = shots["shot_outcome_name"].apply(lambda x: 1 if x == "Goal" else 0)
shots["body_part"] = shots["shot_body_part_name"]
shots["play_type"] = shots["shot_type_name"]
shots["under_pressure"] = shots["under_pressure"].fillna(False)

shots.head()


In [ ]:
import numpy as np

GOAL_X = 120
GOAL_Y = 40

def calculate_distance(x, y):
    return np.sqrt((GOAL_X - x)**2 + (GOAL_Y - y)**2)

def calculate_angle(x, y):
    left_post = np.array([120, 36])
    right_post = np.array([120, 44])
    shot = np.array([x, y])
    
    a = np.linalg.norm(left_post - shot)
    b = np.linalg.norm(right_post - shot)
    c = np.linalg.norm(left_post - right_post)
    
    return np.arccos((a**2 + b**2 - c**2) / (2 * a * b))

shots["distance"] = shots.apply(lambda r: calculate_distance(r.shot_x, r.shot_y), axis=1)
shots["angle"] = shots.apply(lambda r: calculate_angle(r.shot_x, r.shot_y), axis=1)

shots[["shot_x", "shot_y", "distance", "angle", "goal"]].head()


Below is where we start on the freeze-frame part

Selecting the shots with freeze frame and returning the percentage with.

In [ ]:
shots["has_freeze"] = shots["shot_freeze_frame"].notna()
shots["has_freeze"].mean()

Extract the goalkeeper position

In [ ]:
import numpy as np

GOAL_CENTER = np.array([120, 40])

def get_keeper_xy(freeze):
    if not isinstance(freeze, list):
        return (np.nan, np.nan)

    keeper_loc = None
    min_dist = float("inf")

    for p in freeze:
        # Skip attackers
        if p.get("teammate", False):
            continue
        
        loc = np.array(p.get("location", [np.nan, np.nan]))
        dist = np.linalg.norm(loc - GOAL_CENTER)

        if dist < min_dist:
            min_dist = dist
            keeper_loc = loc

    if keeper_loc is None:
        return (np.nan, np.nan)

    return keeper_loc[0], keeper_loc[1]


In [ ]:
shots["keeper_x"], shots["keeper_y"] = zip(*shots["shot_freeze_frame"].apply(get_keeper_xy))

In [ ]:
shots[["keeper_x", "keeper_y"]].head()

Extract the defender position

In [ ]:
def get_defender_locations(freeze, keeper_xy):
    if not isinstance(freeze, list):
        return []

    defenders = []
    kx, ky = keeper_xy
    keeper = np.array([kx, ky])

    for p in freeze:
        if p.get("teammate", False):
            continue  # skip attackers

        loc = np.array(p.get("location", [np.nan, np.nan]))

        # skip goalkeeper
        if np.allclose(loc, keeper, atol=1e-3):
            continue

        defenders.append(loc)

    return defenders


In [ ]:
def nearest_defender_distance(shooter, freeze, keeper_xy):
    defenders = get_defender_locations(freeze, keeper_xy)
    if not defenders:
        return np.nan

    s = np.array(shooter)
    dists = [np.linalg.norm(d - s) for d in defenders]
    return min(dists)

In [ ]:
shots["nearest_defender"] = shots.apply(
    lambda r: nearest_defender_distance(
        (r.shot_x, r.shot_y),
        r.shot_freeze_frame,
        (r.keeper_x, r.keeper_y)
    ),
    axis=1
)

Feature: defender density (within 5m

In [ ]:
def defender_density(shooter, freeze, keeper_xy, radius=5):
    defenders = get_defender_locations(freeze, keeper_xy)
    if not defenders:
        return np.nan

    s = np.array(shooter)
    return sum(np.linalg.norm(d - s) <= radius for d in defenders)


In [ ]:
shots["defender_density"] = shots.apply(
    lambda r: defender_density(
        (r.shot_x, r.shot_y),
        r.shot_freeze_frame,
        (r.keeper_x, r.keeper_y)
    ),
    axis=1
)

Feature: defenders between shooter and goal

A defender is “between” if they lie inside the triangle formed by
shooter → left post → right post


In [ ]:
LEFT_POST = np.array([120, 36])
RIGHT_POST = np.array([120, 44])

def defenders_between(shooter, freeze, keeper_xy):
    defenders = get_defender_locations(freeze, keeper_xy)
    if not defenders:
        return np.nan

    s = np.array(shooter)
    count = 0

    for d in defenders:
        # Barycentric coordinate test for point inside triangle
        v0 = RIGHT_POST - LEFT_POST
        v1 = d - LEFT_POST
        v2 = s - LEFT_POST

        dot00 = np.dot(v0, v0)
        dot01 = np.dot(v0, v1)
        dot02 = np.dot(v0, v2)
        dot11 = np.dot(v1, v1)
        dot12 = np.dot(v1, v2)

        inv_denom = 1 / (dot00 * dot11 - dot01 * dot01 + 1e-9)
        u = (dot11 * dot02 - dot01 * dot12) * inv_denom
        v = (dot00 * dot12 - dot01 * dot02) * inv_denom

        if u >= 0 and v >= 0 and (u + v) <= 1:
            count += 1

    return count

In [ ]:
shots["defenders_between"] = shots.apply(
    lambda r: defenders_between(
        (r.shot_x, r.shot_y),
        r.shot_freeze_frame,
        (r.keeper_x, r.keeper_y)
    ),
    axis=1
)

In [ ]:
shots[[
    "shot_x", "shot_y", "goal",
    "nearest_defender", "defender_density", "defenders_between",
    "keeper_x", "keeper_y"
]].head()

You’ve built a rich, StatsBomb‑quality xG dataset with:
- Distance
- Angle
- Body part
- Play type
- Under pressure
- Nearest defender
- Defender density
- Defenders between
- Goalkeeper position
This is enough to train a very strong xG model.

Generat clean shots data frame

In [ ]:
shots.head()

In [ ]:
clean_cols = [
    "match_id",
    "shot_x", "shot_y",
    "distance", "angle",
    "goal",
    "body_part", "play_type", "under_pressure",
    "keeper_x", "keeper_y",
    "nearest_defender", "defender_density", "defenders_between"
]

shots_clean = shots[clean_cols].copy()


In [ ]:
shots_clean.head()

In [ ]:
shots_clean.to_csv("../data/shots_clean.csv", index=False)